# Resume / document Q&A with Qwen2.5-3B + Unsloth (fixed version)

This replaces `llmswiftfinetuning.ipynb`. The original gave wrong, made-up answers for these reasons:

| # | problem in the original notebook | effect | fix in this notebook |
|---|---|---|---|
| 1 | **Only ONE training example**: a single question mapped to the whole resume dump | The model never learns individual facts, and can't answer *"How many years of experience?"* | Build **hundreds of Q&A pairs** (one fact each, several phrasings) from the document |
| 2 | The resume is ~3,000 tokens but `max_length=2048` → **truncated** | The second half of the resume (IBM, Accenture...) and the closing `<|im_end|>` were cut off, so the model never saw them or learned when to stop | `MAX_SEQ_LENGTH = 8192`, short Q&A examples, token count checked |
| 3 | 15 steps on 1 example, loss still ~2.2 | Far from learned. The answers ("CPA", "Microsoft", "accounting") are pure hallucination | Several epochs over the whole Q&A set, with loss monitored |
| 4 | Loss computed on the **prompt as well as** the answer (`labels = input_ids`) | Training is wasted on the system and user text | Prompt tokens masked with `-100`; only answers are learned |
| 5 | The test questions were **never trained** and the expected answers were **wrong** (Rust, PyTorch and "20 years" are not in the resume) | You couldn't measure accuracy | Corrected test suite with automatic scoring |
| 6 | Hand-written ChatML with a typo `<|im_end|u003E` (cell 1), `split("assistant\n")` to cut the output, `temperature` without `do_sample` | The model learned to print the literal `<|im_end|u003E` text and kept rambling; decoding was fragile | `tokenizer.apply_chat_template`, decode only the new tokens, greedy decoding |
| 7 | `pip install "trl<0.9.0" "xformers<0.0.27" --no-deps` | Ancient pins that clash with current Unsloth / Transformers 5 | `pip install unsloth`, plus plain `transformers.Trainer` so the TRL version doesn't matter |

**Key point:** a 3B model **cannot memorise a document word for word from one example**. Two approaches work, and this notebook does both and compares them:

* **A. Grounded answering**: put the resume (it is small) in the prompt. No training is needed, it is the most accurate option, and it updates instantly.
* **B. Fine-tuning (QLoRA SFT)** on many Q&A pairs generated from the document. The model *knows* the facts without the document in the prompt. Expect it to be good, but less exact than A.
* **Best: B + A.** The fine-tuned model answers in your style, with the document supplied as context.

Runtime: **Colab T4 GPU**. Upload your `.docx` (or `.pdf`) to `/content` first.

## 1. Install

In [ ]:
# Don't pin old trl/xformers versions - they break current Unsloth.
!pip install -q unsloth python-docx pypdf

## 2. Configuration

In [ ]:
DOC_PATH        = "/content/Sifayath_Ahmed_Resume_V6_Designed.docx"   # .docx or .pdf
MODEL_NAME      = "unsloth/Qwen2.5-3B-Instruct"   # smaller: "unsloth/Qwen2.5-1.5B-Instruct", "unsloth/Qwen2.5-0.5B-Instruct"
USE_UNSLOTH     = True        # False -> plain transformers + peft + bitsandbytes (fallback)
MAX_SEQ_LENGTH  = 8192        # whole document must fit for grounded answering (resume ~3k tokens)
LOAD_IN_4BIT    = True        # QLoRA
GENERATE_QA_WITH_LLM = True   # let the base model write extra Q&A pairs from each section
EPOCHS          = 4
LEARNING_RATE   = 2e-4
LORA_R, LORA_ALPHA = 16, 32
MAX_STEPS       = -1          # set e.g. 20 for a quick smoke test
SEED = 3407

## 3. Read the document correctly
Reads paragraphs **and** table cells in document order. Designed resumes put the whole layout in a table, and merged cells are de-duplicated. Also supports PDF.

In [ ]:
import re, json, random
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph

def iter_block_text(parent):
    # paragraphs and tables in the order they appear (recurses into nested tables)
    body = parent.element.body if hasattr(parent, "element") and hasattr(parent.element, "body") else parent._tc
    for child in body.iterchildren():
        tag = child.tag.split('}')[-1]
        if tag == "p":
            yield Paragraph(child, parent).text
        elif tag == "tbl":
            seen = set()
            for row in Table(child, parent).rows:
                for cell in row.cells:
                    if id(cell._tc) in seen:          # merged cells repeat -> skip duplicates
                        continue
                    seen.add(id(cell._tc))
                    yield from iter_block_text(cell)

def read_document(path):
    if path.lower().endswith(".pdf"):
        from pypdf import PdfReader
        text = "\n".join(p.extract_text() or "" for p in PdfReader(path).pages)
    else:
        text = "\n".join(iter_block_text(Document(path)))
    lines = [re.sub(r"\s+", " ", l).strip() for l in text.splitlines()]
    return "\n".join(l for l in lines if l)

doc_text = read_document(DOC_PATH)
print(f"{len(doc_text.split())} words, {len(doc_text)} chars\n")
# preview with contact details masked (so saved notebook outputs don't leak them)
preview = re.sub(r"\+?\d[\d \-]{8,}\d", "[phone]", doc_text[:1500])
print(re.sub(r"\S+@\S+", "[email]", preview))

## 4. Load the base model

In [ ]:
import torch
if USE_UNSLOTH:
    from unsloth import FastLanguageModel        # import unsloth before transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_base():
    if USE_UNSLOTH:
        return FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
                                                 dtype=None, load_in_4bit=LOAD_IN_4BIT)
    kw = {}
    if LOAD_IN_4BIT and torch.cuda.is_available():
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
        kw["device_map"] = "auto"
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    m = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kw)
    if not kw:
        m.to("cuda" if torch.cuda.is_available() else "cpu")
    return m, tok

model, tokenizer = load_base()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
n_tok = len(tokenizer(doc_text).input_ids)
print(f"Document = {n_tok} tokens (the original notebook truncated at 2048!)")
assert n_tok + 600 < MAX_SEQ_LENGTH, "Increase MAX_SEQ_LENGTH or chunk the document"

## 5. Chat helper and test suite
Always use the tokenizer's chat template, decode **only the newly generated tokens**, and use greedy decoding for factual answers.

In [ ]:
NAME = doc_text.splitlines()[0].title()          # first line of the resume = candidate name
REFUSAL = "The resume does not contain that information."
SYSTEM_FT = (f"You are an assistant that answers questions about {NAME}'s resume accurately and concisely. "
             f"If the resume does not contain the answer, reply: '{REFUSAL}'")

def build_messages(question, context=None):
    system = SYSTEM_FT
    if context:
        system += "\n\nAnswer ONLY from this resume:\n<resume>\n" + context + "\n</resume>"
    return [{"role": "system", "content": system}, {"role": "user", "content": question}]

@torch.no_grad()
def ask(question, context=None, max_new_tokens=200):
    if USE_UNSLOTH:
        FastLanguageModel.for_inference(model)
    prompt = tokenizer.apply_chat_template(build_messages(question, context), tokenize=False, add_generation_prompt=True)
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id, use_cache=True)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
# Test questions with expected key facts TAKEN FROM THE RESUME.
# Each entry in `must` is a fact; a list inside means "any of these spellings".
# Edit these for your own document.
TEST_SUITE = [
    {"q": "What is the candidate's primary job title or area of expertise?",
     "must": [["architect"], ["genai", "generative ai", "agentic ai", "enterprise ai"]]},
    {"q": "How many years of professional experience does the candidate have?", "must": [["19"]]},
    {"q": "Which programming languages are listed on the resume?", "must": [["python"], ["pyspark"], ["pl/sql"]]},
    {"q": "Who is the current employer and what is the current role?", "must": [["hcl"], ["enterprise architect"]]},
    {"q": "What is the candidate's educational qualification?", "must": [["electronics"], ["anna university"], ["2006"]]},
    {"q": "Which company did the candidate work for just before HCLTech?", "must": [["ascendion"]]},
    {"q": "What is the largest team size the candidate has led?", "must": [["20"]]},
    {"q": "What did the candidate achieve in TechGig Code Gladiator?", "must": [["finalist"], ["2022"], ["2023"]]},
    {"q": "What was the impact of the task prioritization project?", "must": [["40%", "40 %"], ["45%", "45 %"]]},
    {"q": "Which cloud or LLM platforms has the candidate used?", "must": [["azure openai"], ["vertex ai"], ["databricks"]]},
    {"q": "When did the candidate work at IBM?", "must": [["2014"], ["2022"]]},
    {"q": "What is the architecture of the AI Foundry Agentic Hub based on?", "must": [["langgraph"]]},
    {"q": "What is the candidate's expected salary?", "must": "REFUSE"},
    {"q": "Does the candidate have a PhD from MIT?", "must": "REFUSE"},
]

def norm(s): return s.lower().replace("’", "'").replace("–", "-").replace("—", "-")
REFUSE_RE = re.compile(r"does not (contain|mention|include|specify)|not (mentioned|specified|provided|available)|no information|doesn't (contain|mention)", re.I)

def score(answer, must):
    a = norm(answer)
    if must == "REFUSE":
        return float(bool(REFUSE_RE.search(answer)))
    return sum(any(norm(x) in a for x in alt) for alt in must) / len(must)

def run_suite(label, context=None, show=True):
    scores = []
    for t in TEST_SUITE:
        ans = ask(t["q"], context)
        s = score(ans, t["must"]); scores.append(s)
        if show:
            print(f"[{s:.2f}] Q: {t['q']}\n       A: {ans[:300]}\n")
    acc = sum(scores) / len(scores)
    print(f"==> {label}: {acc:.1%} of expected facts found")
    return acc

RESULTS = {}

## 6. Baseline: base model, no document
This should score poorly. It is what your original notebook was effectively doing, with hallucinated answers.

In [ ]:
RESULTS["base, no document"] = run_suite("base, no document")

## 7. Approach A: grounded answering (document in the prompt, no training)
The resume is about 3k tokens, so it fits in the context easily. This is the **most accurate** option.

In [ ]:
RESULTS["base + document in prompt"] = run_suite("base + document in prompt", context=doc_text)

## 8. Approach B: build a real fine-tuning dataset

1. **Rule-based facts** parsed from the resume structure (sections, `Key: value` skill lines, timeline, projects), each with **several question phrasings**.
2. **LLM-generated Q&A** (optional): the base model reads each section and writes extra grounded Q&A pairs.
3. **Refusal examples**, so the model says "not in the resume" instead of hallucinating.
4. Test-suite questions are **removed** from training, so the evaluation stays honest.

In [ ]:
HEADING_RE = re.compile(r"^[A-Z][A-Z &/()-]{2,}$")          # e.g. CORE SKILLS, CAREER TIMELINE
DATE_RANGE = r"\((?:[A-Z][a-z]{2} )?\d{4}(?:\s*[–-]\s*(?:[A-Z][a-z]{2} )?(?:\d{4}|Present))?\)"   # (Mar 2025 – Present), (2024)

def split_sections(text):
    sections, cur, buf = [], "HEADER", []
    for line in text.splitlines()[1:]:              # line 0 is the name
        if HEADING_RE.match(line) and len(line) < 40:
            sections.append((cur, buf)); cur, buf = line.strip(), []
        else:
            buf.append(line)
    sections.append((cur, buf))
    return [(h, b) for h, b in sections if b]

C_ = ["the candidate", NAME.split()[0]]   # vary the subject in questions
def qs(*templates):
    return [t.format(c=c) for t in templates for c in C_]

def rule_based_facts(text):
    facts = []
    secs = split_sections(text)
    for head, lines in secs:
        body = "\n".join(lines)
        h = head.lower()
        # whole-section questions (skip very long sections - their parts are covered below)
        if len(body.split()) <= 250 and head != "HEADER":
            facts.append({"questions": qs(f"What does {{c}}'s resume list under {h}?", f"Summarize {{c}}'s {h}.",
                                      f"Tell me about {{c}}'s {h}."), "answer": body})
        if head == "HEADER":
            facts.append({"questions": ["What is the candidate's name?", "Whose resume is this?"], "answer": NAME})
            facts.append({"questions": qs("What is {c}'s job title?", "What is {c}'s headline role?",
                                          "What is {c}'s primary area of expertise?"),
                          "answer": " / ".join(lines[:2])})
        # "Key: value" lines (skills etc.)
        for l in lines:
            m = re.match(r"^([A-Za-z][\w /&()-]{2,40}):\s+(.{3,})$", l)
            if m and head != "PROFESSIONAL EXPERIENCE":
                k, v = m.group(1).strip(), m.group(2).strip()
                facts.append({"questions": qs(f"What {k} skills does {{c}} have?", f"List {{c}}'s {k} skills.",
                                              f"Which {k} technologies does {{c}} know?"),
                              "answer": f"{k}: {v}"})
        # timeline lines: "Title, Company — Mon YYYY to Present"
        timeline = []
        for l in lines:
            m = re.match(r"^([^,]{3,60}), ([^,]{2,60}) [—-] ([A-Z][a-z]{2} \d{4}) to ([A-Z][a-z]{2} \d{4}|Present)$", l)
            if not m:
                continue
            timeline.append(m.groups())
            title, comp, start, end = m.groups()
            ans = f"{title} at {comp}, from {start} to {end}."
            facts.append({"questions": qs(f"When did {{c}} work at {comp}?", f"What was {{c}}'s role at {comp}?",
                                          f"How long was {{c}} at {comp}?"), "answer": ans})
            if end.lower() == "present":
                facts.append({"questions": qs("Where does {c} currently work?", "What is {c}'s current role?",
                                              "Who is {c}'s current employer?"), "answer": ans})
        # timeline is newest-first: derive "worked before X" facts
        for (t1, c1, *_), (t0, c0, s0, e0) in zip(timeline, timeline[1:]):
            facts.append({"questions": qs(f"Which company did {{c}} work for before {c1}?",
                                          f"Where did {{c}} work prior to joining {c1}?"),
                          "answer": f"Before {c1}, {NAME.split()[0]} worked at {c0} as {t0} ({s0} to {e0})."})
        # experience projects: heading line ending in a date range, followed by bullets and "Tech:"
        if head == "PROFESSIONAL EXPERIENCE":
            cur, bullets, tech = None, [], None
            def flush():
                if cur and bullets:
                    name = re.sub(DATE_RANGE, "", cur).strip(" ▸")
                    facts.append({"questions": qs(f"What did {{c}} do in the {name} project?",
                                                  f"Describe {{c}}'s work on {name}.",
                                                  f"What were {{c}}'s responsibilities in {name}?"),
                                  "answer": cur.strip(" ▸") + "\n- " + "\n- ".join(bullets)})
                    if tech:
                        facts.append({"questions": qs(f"What technologies were used in {name}?",
                                                      f"Which tech stack did {{c}} use for {name}?"),
                                      "answer": f"{name}: {tech}"})
            for l in lines:
                if re.search(DATE_RANGE + r"\s*$", l):
                    flush(); cur, bullets, tech = l, [], None
                elif l.startswith("Tech:"):
                    tech = l
                elif cur:
                    bullets.append(l)
            flush()
    return facts

# list items in short sections (achievements, highlights, domains) -> one fact each
def list_item_facts(text):
    facts = []
    for head, lines in split_sections(text):
        if head in {"ACHIEVEMENTS", "CAREER HIGHLIGHTS", "PROFILE"}:
            for l in lines:
                key = " ".join(l.split()[:6])
                facts.append({"questions": [f"What does the resume say about '{key}'?",
                                            f"Tell me more about this item from the {head.lower()}: {key}"],
                              "answer": l})
    return facts

facts = rule_based_facts(doc_text) + list_item_facts(doc_text)
print(len(facts), "rule-based facts")
print(json.dumps(facts[5], indent=1)[:600])

In [ ]:
# Optional: let the base model write extra grounded Q&A pairs for each section (synthetic data).
QA_PROMPT = '''Read this section of a resume and write {n} question-answer pairs a recruiter might ask.
Rules: answers must be fully supported by the text, specific (names, numbers, dates, technologies), 1-3 sentences.
For each question also give 2 paraphrases. Return ONLY a JSON list:
[{{"question": "...", "paraphrases": ["...", "..."], "answer": "..."}}]

SECTION ({head}):
{body}'''

def llm_facts(text, n=6):
    out = []
    for head, lines in split_sections(text):
        body = "\n".join(lines)
        # split long sections into ~350-word chunks
        words, chunks, cur = body.split("\n"), [], []
        for w in words:
            cur.append(w)
            if sum(len(x.split()) for x in cur) > 350:
                chunks.append("\n".join(cur)); cur = []
        if cur: chunks.append("\n".join(cur))
        for ch in chunks:
            raw = ask_raw(QA_PROMPT.format(n=n, head=head, body=ch))
            m = re.search(r"\[.*\]", raw, re.S)
            try:
                items = json.loads(m.group(0)) if m else []
            except json.JSONDecodeError:
                items = []
            for it in items:
                if not isinstance(it, dict) or not it.get("question") or not it.get("answer"):
                    continue
                ans = str(it["answer"]).strip()
                # keep only answers whose numbers all appear in the document (cheap hallucination filter)
                if any(num not in text for num in re.findall(r"\d+", ans)):
                    continue
                qs_ = [it["question"]] + [p for p in it.get("paraphrases", []) if isinstance(p, str)]
                out.append({"questions": qs_, "answer": ans})
            print(f"{head[:25]:25s} -> {len(out)} LLM facts so far")
    return out

@torch.no_grad()
def ask_raw(prompt, max_new_tokens=1200):
    if USE_UNSLOTH:
        FastLanguageModel.for_inference(model)
    p = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    enc = tokenizer(p, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)

if GENERATE_QA_WITH_LLM:
    facts += llm_facts(doc_text)
print(len(facts), "facts total")

In [ ]:
# Refusal examples (questions the resume cannot answer)
OUT_OF_SCOPE = [
    "What is the candidate's date of birth?", "What is the candidate's current salary?",
    "Is the candidate married?", "What is the candidate's notice period?", "What is the candidate's religion?",
    "Which university did the candidate do a master's degree at?", "What is the candidate's home address?",
    "Does the candidate have a driving licence?", "What is the candidate's favourite movie?",
    "How tall is the candidate?", "What is the candidate's blood group?", "What are the candidate's hobbies?",
    "What is the capital of France?", "Write a poem about the ocean.", "Who won the cricket world cup in 2011?",
]
test_qs = {norm(t["q"]) for t in TEST_SUITE}

rng = random.Random(SEED)
train_rows, val_rows = [], []
for f in facts:
    qs_ = [q for q in dict.fromkeys(f["questions"]) if norm(q) not in test_qs]
    rng.shuffle(qs_)
    rows = [{"q": q, "a": f["answer"]} for q in qs_]
    if len(rows) >= 3:                 # hold out one phrasing per fact for validation loss
        val_rows.append(rows[0]); train_rows += rows[1:]
    else:
        train_rows += rows
train_rows += [{"q": q, "a": REFUSAL} for q in OUT_OF_SCOPE if norm(q) not in test_qs]
rng.shuffle(train_rows)
print(f"train={len(train_rows)}  val={len(val_rows)}")
train_rows[:3]

## 9. Tokenise with answer-only loss
The prompt (system + question) gets label `-100`, so the model is trained **only on the answer tokens**, including the closing `<|im_end|>`, which teaches it when to stop.

In [ ]:
from datasets import Dataset

def encode(row):
    msgs = build_messages(row["q"])                                   # no context: knowledge must be in the weights
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    full = tokenizer.apply_chat_template(msgs + [{"role": "assistant", "content": row["a"]}], tokenize=False)
    assert full.startswith(prompt)
    p_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tokenizer(full, add_special_tokens=False)["input_ids"]
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids), "labels": labels}

# Drop (don't truncate) over-long examples: truncation would cut off <|im_end|> and the model never learns to stop.
MAX_EXAMPLE_TOKENS = 1024
fits = lambda ex: len(ex["input_ids"]) <= MAX_EXAMPLE_TOKENS
train_ds = Dataset.from_list(train_rows).map(encode, remove_columns=["q", "a"]).filter(fits)
val_ds = Dataset.from_list(val_rows).map(encode, remove_columns=["q", "a"]).filter(fits) if val_rows else None
ex = train_ds[0]
print("Trained-on text:", repr(tokenizer.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])))
print("Longest example:", max(len(x) for x in train_ds["input_ids"]), "tokens")

## 10. Add LoRA adapters and train

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
if USE_UNSLOTH:
    model = FastLanguageModel.get_peft_model(model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0,
                                             target_modules=TARGETS, bias="none",
                                             use_gradient_checkpointing="unsloth", random_state=SEED)
    FastLanguageModel.for_training(model)
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if LOAD_IN_4BIT and torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
                                             target_modules=TARGETS, task_type="CAUSAL_LM"))
model.print_trainable_parameters()

bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir="outputs_resume",
    num_train_epochs=EPOCHS, max_steps=MAX_STEPS,
    per_device_train_batch_size=4, gradient_accumulation_steps=2, per_device_eval_batch_size=4,
    learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_steps=10, weight_decay=0.0,
    fp16=torch.cuda.is_available() and not bf16, bf16=bf16,
    optim="adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    logging_steps=10, eval_strategy="epoch" if val_ds is not None else "no",
    save_strategy="no", report_to="none", seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
trainer.train()
# Healthy run: train loss falls well below 0.5. If eval_loss starts rising while train loss falls, reduce EPOCHS.

## 11. Evaluate the fine-tuned model

In [ ]:
RESULTS["fine-tuned, no document"] = run_suite("fine-tuned, no document")

In [ ]:
RESULTS["fine-tuned + document in prompt"] = run_suite("fine-tuned + document in prompt", context=doc_text)

In [ ]:
import pandas as pd
pd.DataFrame({"setup": list(RESULTS), "facts found": [f"{v:.0%}" for v in RESULTS.values()]})

## 12. Ask your own questions

In [ ]:
for q in ["Summarize the candidate's experience in 3 bullet points.",
          "Has the candidate worked with MCP and Google A2A?",
          "What guardrails did the candidate build?"]:
    print("Q:", q)
    print("A (fine-tuned):          ", ask(q), "\n")
    print("A (fine-tuned + resume): ", ask(q, context=doc_text), "\n")

## 13. Save and reload
Save the **adapter** (not a mid-training checkpoint) and reload it with the same loader.

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
# Reload later:
# model, tokenizer = FastLanguageModel.from_pretrained("lora_model", max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
# FastLanguageModel.for_inference(model)

## 14. Export a merged model for vLLM / Transformers
Merges the LoRA adapter into the base weights and saves a normal 16-bit model folder (`merged_model/`).
vLLM, TGI, Transformers and the GGUF converter all load this folder.

In [ ]:
import gc
MERGED_DIR = "merged_model"
if USE_UNSLOTH:
    model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
else:
    # Merging into 4-bit weights is lossy, so reload the base in fp16 and apply the saved adapter.
    from peft import PeftModel
    model.save_pretrained("lora_model"); tokenizer.save_pretrained("lora_model")
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float16)
    merged = PeftModel.from_pretrained(base, "lora_model").merge_and_unload()
    merged.save_pretrained(MERGED_DIR); tokenizer.save_pretrained(MERGED_DIR)
    del base, merged; gc.collect()

# Save the system prompt next to the model: the model was trained with it, so use it at inference too.
open(f"{MERGED_DIR}/system_prompt.txt", "w").write(SYSTEM_FT)
!ls -lh {MERGED_DIR}

## 15. Export a GGUF file (llama.cpp / Ollama / LM Studio)
* `q4_k_m`: about 1 GB for a 1.5B model. Small and fast, with a slight quality loss. **Recommended.**
* `q8_0`: about 1.6 GB, almost lossless. `f16`: full precision.

Unsloth builds llama.cpp the first time it runs, which takes a few minutes. If that fails, the cell falls back to
llama.cpp's converter on `merged_model/`, which writes `q8_0` without compiling anything.

In [ ]:
import glob, os, subprocess
GGUF_QUANT = "q4_k_m"          # "q8_0", "f16"
GGUF_DIR = "gguf_model"
gguf_path = None

if USE_UNSLOTH:
    try:
        model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=GGUF_QUANT)
        # file name and folder differ between Unsloth versions -> search for it
        found = glob.glob(f"{GGUF_DIR}*/**/*.gguf", recursive=True) + glob.glob("*.gguf")
        preferred = [f for f in found if GGUF_QUANT.replace("_", "").lower() in os.path.basename(f).replace("_", "").lower()]
        if found:   # give it a predictable name
            gguf_path = f"resume-bot-{GGUF_QUANT}.gguf"
            os.replace(max(preferred or found, key=os.path.getmtime), gguf_path)
    except Exception as e:
        print("Unsloth GGUF export failed -> using llama.cpp converter instead:", e)

if gguf_path is None:
    if not os.path.exists("llama.cpp"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp"], check=True)
    subprocess.run(["pip", "install", "-q", "gguf", "sentencepiece"], check=True)
    gguf_path = "resume-bot-q8_0.gguf"
    subprocess.run(["python", "llama.cpp/convert_hf_to_gguf.py", MERGED_DIR,
                    "--outfile", gguf_path, "--outtype", "q8_0"], check=True)

print(f"GGUF file: {gguf_path}  ({os.path.getsize(gguf_path) / 1e9:.2f} GB)")

# Ollama: ollama create resume-bot -f Modelfile   then   ollama run resume-bot
with open("Modelfile", "w") as f:
    f.write(f'FROM ./{os.path.basename(gguf_path)}\nPARAMETER temperature 0\nPARAMETER num_ctx 8192\nSYSTEM """{SYSTEM_FT}"""\n')
print(open("Modelfile").read())

## 16. Use the GGUF model to answer questions about the resume
This uses [llama-cpp-python](https://github.com/abetlen/llama-cpp-python). It runs on CPU, so the same code works on your laptop.
The chat template is stored inside the GGUF file. Use the **same system prompt** as in training.
Answers grounded in the resume (`context=doc_text`) are the most accurate, but the 3k-token prompt is slower on CPU.

In [ ]:
# Prebuilt CPU wheel (no compiling). For GPU: use .../whl/cu124 and n_gpu_layers=-1 below.
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

In [ ]:
import os
from llama_cpp import Llama

gguf_llm = Llama(model_path=gguf_path, n_ctx=8192, n_threads=os.cpu_count(), n_gpu_layers=0, verbose=False)

def ask_gguf(question, context=None, max_tokens=200):
    out = gguf_llm.create_chat_completion(messages=build_messages(question, context),
                                          temperature=0.0, max_tokens=max_tokens)
    return out["choices"][0]["message"]["content"].strip()

for q in ["Where does the candidate currently work, and in what role?",
          "Which programming languages does the candidate know?",
          "What is the candidate's expected salary?"]:
    print("Q:", q)
    print("A (GGUF):           ", ask_gguf(q))
    print("A (GGUF + resume):  ", ask_gguf(q, context=doc_text), "\n")

### Standalone script (on your laptop, without this notebook)
Copy `resume-bot*.gguf` (or the file printed above) and `merged_model/system_prompt.txt`, then:
```python
# pip install llama-cpp-python
from llama_cpp import Llama
llm = Llama(model_path="resume-bot-q4_k_m.gguf", n_ctx=8192, verbose=False)
system = open("system_prompt.txt").read()
r = llm.create_chat_completion(messages=[{"role": "system", "content": system},
                                         {"role": "user", "content": "Where does the candidate currently work?"}],
                               temperature=0, max_tokens=200)
print(r["choices"][0]["message"]["content"])
```

## 17. (Optional) Serve the merged model with vLLM
vLLM needs the GPU to itself and conflicts with Unsloth's packages, so run this in a **fresh runtime**:
*Runtime -> Disconnect and delete runtime*, then run only the cell below with `RUN_VLLM = True`. It needs `merged_model/` to still exist,
so first copy it to Google Drive (next section) and copy it back. On a T4 use `dtype="half"`, because T4s don't support bf16.
OpenAI-compatible server alternative: `vllm serve merged_model --dtype half --max-model-len 8192`

In [ ]:
RUN_VLLM = False
if RUN_VLLM:
    # !pip install -q vllm
    from vllm import LLM, SamplingParams
    system = open("merged_model/system_prompt.txt").read()
    llm = LLM(model="merged_model", dtype="half", max_model_len=8192, gpu_memory_utilization=0.85)
    questions = ["Where does the candidate currently work?", "What is the candidate's education?"]
    outs = llm.chat([[{"role": "system", "content": system}, {"role": "user", "content": q}] for q in questions],
                    SamplingParams(temperature=0, max_tokens=200))
    for q, o in zip(questions, outs):
        print("Q:", q, "\nA:", o.outputs[0].text, "\n")

## 18. Download the files / copy to Google Drive

In [ ]:
import shutil
shutil.make_archive("merged_model", "zip", "merged_model")      # for vLLM
try:
    from google.colab import files
    files.download(gguf_path)                                    # GGUF file
    files.download("Modelfile")
    # files.download("merged_model.zip")                         # large (~3 GB for 1.5B)
    # Or keep them in Drive:
    # from google.colab import drive; drive.mount("/content/drive")
    # !cp -r {gguf_path} merged_model Modelfile /content/drive/MyDrive/
except ImportError:
    print("Not running in Colab - files are in:", os.getcwd())

## Reading the results / tips
* **Base, no document**: low score with hallucinations. This is the behaviour you saw.
* **Base + document**: highest accuracy. For a single resume, grounding alone is usually enough.
* **Fine-tuned, no document**: should be far better than the base model, with some details (exact dates, percentages) still missed. Fine-tuning teaches *knowledge approximately* and *behaviour exactly*.
* **Fine-tuned + document**: the best of both. Use it in production.

If the fine-tuned score is low:
1. Train more: `EPOCHS = 6-8`, or raise `LORA_R = 32` with `LORA_ALPHA = 64`.
2. Add more phrasings per fact, and real recruiter questions.
3. Check the generated Q&A (`train_rows`) for wrong answers. Garbage in, garbage out.
4. For many or large documents, use retrieval (RAG) instead of the whole document. See `src/rag.py` in this repo.